# Large-Scale Benchmark
Full dataset loaded from HuggingFace, with processing cost (`proc_cost`) attached.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

/Users/yattmeo/Desktop/SMU/Code/404_found_us/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Raw Dataset from HuggingFace

In [2]:
ds = load_dataset('thiru1711/Financial_Transactions')
print(ds)

DatasetDict({
    train: Dataset({
        features: ['transaction_id', 'date', 'card_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'is_fraud', 'card_brand', 'card_type', 'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date', 'year_pin_last_changed', 'card_on_dark_web', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards', 'mcc_description'],
        num_rows: 13305915
    })
})


In [3]:
splits = [ds[split].to_pandas() for split in ds.keys()]
raw_df = pd.concat(splits, ignore_index=True)
print(f'Total rows : {len(raw_df):,}')
print(f'Columns    : {list(raw_df.columns)}')
raw_df.info()

Total rows : 13,305,915
Columns    : ['transaction_id', 'date', 'card_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'is_fraud', 'card_brand', 'card_type', 'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date', 'year_pin_last_changed', 'card_on_dark_web', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards', 'mcc_description']
<class 'pandas.DataFrame'>
RangeIndex: 13305915 entries, 0 to 13305914
Data columns (total 37 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   transaction_id         str           
 1   date                   datetime64[ns]
 2   card_id                str           
 3   amount                 float32       
 4   use_chip               str           
 5   merchant_id          

## 2. Clean & Standardise

In [4]:
df = raw_df.copy()

# Parse date
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Coerce numeric columns
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
df['mcc']    = pd.to_numeric(df['mcc'],    errors='coerce').astype('Int64')

# Standardise card type label
df['card_type'] = df['card_type'].str.strip().str.lower()

# Drop rows with errors flagged
if 'errors' in df.columns:
    df = df[df['errors'].isna() | (df['errors'] == '')]

# Keep only Visa / Mastercard (match cost-table coverage)
df = df[df['card_brand'].str.lower().isin(['visa', 'mastercard'])]

# drop rows with negative or zero amount
df = df[df['amount'] > 0]

df = df.dropna(subset=['amount', 'mcc', 'date']).reset_index(drop=True)
print(f'Rows after cleaning: {len(df):,}')

Rows after cleaning: 11,324,082


## 3. Attach Processing Cost (`proc_cost`)

In [5]:
from pathlib import Path

# Resolve cost CSV from common notebook working directories
candidate_paths = [
    Path('cost_type_id_18feb.csv'),
    Path('../Clustering/Dataset Creation/cost_type_id_18feb.csv'),
    Path('../../Clustering/Dataset Creation/cost_type_id_18feb.csv'),
    Path('/Users/yattmeo/Desktop/SMU/Code/404_found_us/ml_pipeline/Matt_EDA/Clustering/Dataset Creation/cost_type_id_18feb.csv'),
]

COST_CSV = next((str(p) for p in candidate_paths if p.exists()), None)
if COST_CSV is None:
    raise FileNotFoundError('Could not locate cost_type_id_18feb.csv in expected locations.')

cost_df = pd.read_csv(COST_CSV)
cost_df.columns = [str(c).strip().lower() for c in cost_df.columns]

# Normalize column naming
if 'cost_type_id' not in cost_df.columns and 'cost_typeid' in cost_df.columns:
    cost_df = cost_df.rename(columns={'cost_typeid': 'cost_type_id'})
if 'min_tranaction_amt' in cost_df.columns and 'min_transaction_amt' not in cost_df.columns:
    cost_df = cost_df.rename(columns={'min_tranaction_amt': 'min_transaction_amt'})

required_cols = [
    'cost_type_id', 'card_network', 'card_brand', 'mcc',
    'min_transaction_amt', 'max_transaction_amt',
    'subtotal_fee_percent', 'subtotal_fee_dollars'
 ]
missing = [c for c in required_cols if c not in cost_df.columns]
if missing:
    raise ValueError(f'Missing required cost table columns: {missing}. Available: {list(cost_df.columns)}')

# Parse fee and range columns
for col in [
    'subtotal_fee_percent', 'card_fee_percent', 'network_fee_percent',
    'subtotal_fee_dollars', 'card_fee_dollars', 'network_fee_dollars',
    'min_transaction_amt', 'max_transaction_amt', 'mcc', 'cost_type_id'
 ]:
    if col in cost_df.columns:
        cost_df[col] = (
            cost_df[col].astype(str)
            .str.replace('%', '', regex=False)
            .str.replace('$', '', regex=False)
            .str.strip()
        )
        cost_df[col] = pd.to_numeric(cost_df[col], errors='coerce')

# Build normalized keys for joining
cost_df['network_key'] = cost_df['card_network'].astype(str).str.strip().str.lower()
cost_df['type_key'] = cost_df['card_brand'].astype(str).str.strip().str.lower()
cost_df['mcc'] = cost_df['mcc'].astype('Int64')

# Standardize transaction-side keys
df['card_brand_key'] = df['card_brand'].astype(str).str.strip().str.lower()
df['card_type_key'] = df['card_type'].astype(str).str.strip().str.lower()
df['card_type_key'] = df['card_type_key'].replace({'debit (prepaid)': 'prepaid'})

# Keep naming consistent with downstream cells
cost_df['cost_type_ID'] = cost_df['cost_type_id']

print('Using cost CSV:', COST_CSV)
print(f'Cost table rows: {len(cost_df)}')
print('Cost table columns:', list(cost_df.columns))
cost_df[['cost_type_ID', 'network_key', 'type_key', 'mcc', 'min_transaction_amt', 'max_transaction_amt']].head(5)

Using cost CSV: cost_type_id_18feb.csv
Cost table rows: 62
Cost table columns: ['cost_type_id', 'card_network', 'card_brand', 'fee_program', 'min_transaction_amt', 'max_transaction_amt', 'mcc', 'card_fee_percent', 'card_fee_dollars', 'network_fee_percent', 'network_fee_dollars', 'subtotal_fee_percent', 'subtotal_fee_dollars', 'unnamed: 13', 'network_key', 'type_key', 'cost_type_ID']


,cost_type_ID,network_key,type_key,mcc,min_transaction_amt,max_transaction_amt
0,1.0,visa,prepaid,<NA>,0.000,5.000
1,2.0,visa,debit,<NA>,0.000,5.000
2,3.0,visa,credit,<NA>,0.000,1.818
3,4.0,visa,credit,<NA>,1.818,5.000
4,5.0,visa,super premium credit,<NA>,0.000,1.818


In [6]:
# drop all MCCs except for 5499, 4121, 5411, 5812

df = df[df['mcc'].isin([5499, 4121, 5411, 5812])]

df.info()

<class 'pandas.DataFrame'>
Index: 3880158 entries, 5 to 11324080
Data columns (total 39 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   transaction_id         str           
 1   date                   datetime64[ns]
 2   card_id                str           
 3   amount                 float32       
 4   use_chip               str           
 5   merchant_id            int64         
 6   merchant_city          str           
 7   merchant_state         str           
 8   zip                    float64       
 9   mcc                    Int64         
 10  errors                 str           
 11  is_fraud               int64         
 12  card_brand             str           
 13  card_type              str           
 14  card_number            int64         
 15  expires                str           
 16  cvv                    int16         
 17  has_chip               str           
 18  num_cards_issued       int64         

In [7]:
# cost_df = cost_df[cost_df['mcc'].isin([5499, 4121, 5411, 5812])]

cost_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   cost_type_id          61 non-null     float64
 1   card_network          61 non-null     str    
 2   card_brand            61 non-null     str    
 3   fee_program           61 non-null     str    
 4   min_transaction_amt   61 non-null     float64
 5   max_transaction_amt   61 non-null     float64
 6   mcc                   51 non-null     Int64  
 7   card_fee_percent      61 non-null     float64
 8   card_fee_dollars      61 non-null     float64
 9   network_fee_percent   61 non-null     float64
 10  network_fee_dollars   61 non-null     float64
 11  subtotal_fee_percent  61 non-null     float64
 12  subtotal_fee_dollars  61 non-null     float64
 13  unnamed: 13           0 non-null      float64
 14  network_key           61 non-null     str    
 15  type_key              61 non-null   

In [8]:
# Two-tier match strategy:
# 1) MCC-specific rules for amount > 5
# 2) All-MCC small-ticket rules for remaining rows with amount <= 5

# If earlier filtering removed All-MCC rows, reload full table for small-ticket fallback
cost_source = cost_df.copy()
if cost_source['mcc'].isna().sum() == 0:
    cost_source = pd.read_csv(COST_CSV)
    cost_source.columns = [str(c).strip().lower() for c in cost_source.columns]
    if 'min_tranaction_amt' in cost_source.columns and 'min_transaction_amt' not in cost_source.columns:
        cost_source = cost_source.rename(columns={'min_tranaction_amt': 'min_transaction_amt'})
    for col in [
        'subtotal_fee_percent', 'card_fee_percent', 'network_fee_percent',
        'subtotal_fee_dollars', 'card_fee_dollars', 'network_fee_dollars',
        'min_transaction_amt', 'max_transaction_amt', 'mcc', 'cost_type_id'
    ]:
        if col in cost_source.columns:
            cost_source[col] = (
                cost_source[col].astype(str)
                .str.replace('%', '', regex=False)
                .str.replace('$', '', regex=False)
                .str.strip()
            )
            cost_source[col] = pd.to_numeric(cost_source[col], errors='coerce')

    cost_source['network_key'] = cost_source['card_network'].astype(str).str.strip().str.lower()
    cost_source['type_key'] = cost_source['card_brand'].astype(str).str.strip().str.lower()
    cost_source['mcc'] = cost_source['mcc'].astype('Int64')
    cost_source['cost_type_ID'] = cost_source['cost_type_id']

# Build rule tables
rules_specific = cost_source[[
    'cost_type_ID', 'network_key', 'type_key', 'mcc',
    'min_transaction_amt', 'max_transaction_amt',
    'subtotal_fee_percent', 'subtotal_fee_dollars'
]].dropna(subset=['network_key', 'type_key', 'mcc', 'min_transaction_amt', 'max_transaction_amt'])

rules_general = cost_source[[
    'cost_type_ID', 'network_key', 'type_key', 'mcc',
    'min_transaction_amt', 'max_transaction_amt',
    'subtotal_fee_percent', 'subtotal_fee_dollars'
]].copy()
rules_general = rules_general[rules_general['mcc'].isna()]
rules_general = rules_general.dropna(subset=['network_key', 'type_key', 'min_transaction_amt', 'max_transaction_amt'])

rules_specific = rules_specific.drop_duplicates([
    'network_key', 'type_key', 'mcc', 'min_transaction_amt', 'max_transaction_amt', 'cost_type_ID'
 ]).reset_index(drop=True)
rules_general = rules_general.drop_duplicates([
    'network_key', 'type_key', 'min_transaction_amt', 'max_transaction_amt', 'cost_type_ID'
 ]).reset_index(drop=True)

# Default output columns
df['cost_type_ID'] = np.nan
df['subtotal_fee_percent'] = np.nan
df['subtotal_fee_dollars'] = np.nan

# Pass 1: exact MCC rules for amount > 5
for r in rules_specific.itertuples(index=False):
    mask = (
        (df['card_brand_key'] == r.network_key)
        & (df['card_type_key'] == r.type_key)
        & (df['mcc'] == r.mcc)
        & (df['amount'] > 5)
        & (df['amount'] >= r.min_transaction_amt)
        & (df['amount'] <= r.max_transaction_amt)
    )
    take = mask & df['cost_type_ID'].isna()
    if take.any():
        df.loc[take, 'cost_type_ID'] = r.cost_type_ID
        df.loc[take, 'subtotal_fee_percent'] = r.subtotal_fee_percent
        df.loc[take, 'subtotal_fee_dollars'] = r.subtotal_fee_dollars

# Pass 2: small-ticket All-MCC fallback for amount <= 5
for r in rules_general.itertuples(index=False):
    mask = (
        (df['card_brand_key'] == r.network_key)
        & (df['card_type_key'] == r.type_key)
        & (df['amount'] <= 5)
        & (df['amount'] >= r.min_transaction_amt)
        & (df['amount'] <= r.max_transaction_amt)
    )
    take = mask & df['cost_type_ID'].isna()
    if take.any():
        df.loc[take, 'cost_type_ID'] = r.cost_type_ID
        df.loc[take, 'subtotal_fee_percent'] = r.subtotal_fee_percent
        df.loc[take, 'subtotal_fee_dollars'] = r.subtotal_fee_dollars

print(f'Rows after cost join : {len(df):,}')
print(f'Missing cost_type_ID : {df["cost_type_ID"].isna().sum():,}')
print('Match rate          :', f"{(1 - df['cost_type_ID'].isna().mean())*100:.2f}%")
print(f'Specific rules used : {len(rules_specific):,}')
print(f'General rules used  : {len(rules_general):,}')

Rows after cost join : 3,880,158
Missing cost_type_ID : 0
Match rate          : 100.00%
Specific rules used : 51
General rules used  : 10


In [9]:
# Compute processing cost
df['proc_cost'] = (
    df['subtotal_fee_dollars'].fillna(0)
    + df['subtotal_fee_percent'].fillna(0) * df['amount']
)
print(f'proc_cost NaN count : {df["proc_cost"].isna().sum():,}')
print(df['proc_cost'].describe().round(4))

proc_cost NaN count : 0
count    3.880158e+06
mean     3.442920e+01
std      4.958460e+01
min      6.140000e-02
25%      2.894000e+00
50%      1.206510e+01
75%      4.610360e+01
max      1.310665e+03
Name: proc_cost, dtype: float64


## 4. Dataset Overview

In [10]:
print('=== Dataset Overview ===')
print(f"Date range      : {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Total rows      : {len(df):,}")
print(f"Unique merchants: {df['merchant_id'].nunique():,}")
print(f"Unique MCCs     : {df['mcc'].nunique():,}")
print()
print('--- Transactions per year ---')
print(df.groupby(df['date'].dt.year).size().to_string())
print()
print('--- Top 10 MCCs by volume ---')
print(df['mcc'].value_counts().head(10).to_string())

=== Dataset Overview ===
Date range      : 2010-01-01 to 2019-10-31
Total rows      : 3,880,158
Unique merchants: 17,539
Unique MCCs     : 4

--- Transactions per year ---
date
2010    357898
2011    374146
2012    384623
2013    393740
2014    397284
2015    404694
2016    406766
2017    410683
2018    410230
2019    340094

--- Top 10 MCCs by volume ---
mcc
5411    1446227
5499    1063100
5812     907825
4121     463006


In [11]:
df = df[['transaction_id','merchant_id','date','amount','mcc','card_brand','card_type','cost_type_ID','proc_cost']]

df.head()

,transaction_id,merchant_id,date,amount,mcc,card_brand,card_type,cost_type_ID,proc_cost
5,7475334,59935,2010-01-01 00:09:00,77.000000,5499,Mastercard,debit (prepaid),31.0,133.390000
10,7475339,75781,2010-01-01 00:23:00,2.580000,5411,Visa,debit,2.0,0.694400
11,7475340,59935,2010-01-01 00:26:00,39.630001,5499,Mastercard,debit (prepaid),31.0,68.739902
12,7475341,33326,2010-01-01 00:27:00,43.330002,4121,Visa,debit,12.0,8.029400
19,7475349,21586,2010-01-01 00:37:00,20.740000,5411,Mastercard,credit,36.0,36.010200


In [ ]:
df_5499 = df[df['mcc'] == 5499]
df_4121 = df[df['mcc'] == 4121]
df_5411 = df[df['mcc'] == 5411]
df_5812 = df[df['mcc'] == 5812]

for d in [df_5499, df_4121, df_5411, df_5812]:
    print(f'MCC {d["mcc"].iloc[0]}: {len(d):,} rows')
    

# group each mcc by year and 52 weekly periods


MCC 5499: 1,063,100 rows
MCC 4121: 463,006 rows
MCC 5411: 1,446,227 rows
MCC 5812: 907,825 rows


In [13]:
# Add year and 52-week index columns to each MCC DataFrame
df_list = [df_5499, df_4121, df_5411, df_5812]
updated = []

for d in df_list:
    d = d.copy()
    d['year'] = d['date'].dt.year.astype('Int64')

    # Convert day-of-year to a fixed 1..52 week index
    d['week'] = (((d['date'].dt.dayofyear - 1) // 7) + 1).clip(upper=52).astype('Int64')
    updated.append(d)

df_5499, df_4121, df_5411, df_5812 = updated

for d in [df_5499, df_4121, df_5411, df_5812]:
    print(f"MCC {int(d['mcc'].iloc[0])}: years={d['year'].min()}..{d['year'].max()}, weeks={d['week'].min()}..{d['week'].max()}")

# Optional: also add to the combined df
df = df.copy()
df['year'] = df['date'].dt.year.astype('Int64')
df['week'] = (((df['date'].dt.dayofyear - 1) // 7) + 1).clip(upper=52).astype('Int64')

df[['transaction_id', 'mcc', 'date', 'year', 'week']].head()

MCC 5499: years=2010..2019, weeks=1..52
MCC 4121: years=2010..2019, weeks=1..52
MCC 5411: years=2010..2019, weeks=1..52
MCC 5812: years=2010..2019, weeks=1..52


,transaction_id,mcc,date,year,week
5,7475334,5499,2010-01-01 00:09:00,2010,1
10,7475339,5411,2010-01-01 00:23:00,2010,1
11,7475340,5499,2010-01-01 00:26:00,2010,1
12,7475341,4121,2010-01-01 00:27:00,2010,1
19,7475349,5411,2010-01-01 00:37:00,2010,1


## 5. Building Weekly features

In [14]:
# Build weekly merchant-level feature DataFrames for each MCC (no train/val/test split)
COST_TYPE_IDS = list(range(1, 62))

def build_weekly_merchant_features(df_in, split_year):
    df_local = df_in.copy()
    df_local['date'] = pd.to_datetime(df_local['date'])
    df_local = df_local[df_local['date'].dt.year == split_year].copy()

    # Force an exact 52-week calendar inside each split year
    year_start = pd.Timestamp(f'{split_year}-01-01')
    day_offset = (df_local['date'] - year_start).dt.days
    df_local['week_of_year'] = np.clip((day_offset // 7) + 1, 1, 52).astype(int)

    df_local['cost_type_ID'] = pd.to_numeric(df_local['cost_type_ID'], errors='coerce').astype('Int64')
    df_local['txn_cost_percent'] = np.where(
        df_local['amount'] != 0,
        df_local['proc_cost'] / df_local['amount'],
        np.nan,
    )

    weekly_summary = (
        df_local.groupby(['merchant_id', 'week_of_year'], as_index=False)
        .agg(
            total_proc_cost=('proc_cost', 'sum'),
            total_transaction_value=('amount', 'sum'),
            total_transaction_count=('transaction_id', 'count'),
            cost_percent_stdev=('txn_cost_percent', 'std'),
        )
    )

    weekly_summary['cost_percent'] = np.where(
        weekly_summary['total_transaction_value'] != 0,
        weekly_summary['total_proc_cost'] / weekly_summary['total_transaction_value'],
        np.nan,
    )
    weekly_summary = weekly_summary.drop(columns=['total_proc_cost'])

    cost_type_counts = (
        df_local.dropna(subset=['cost_type_ID'])
        .groupby(['merchant_id', 'week_of_year', 'cost_type_ID'])['transaction_id']
        .count()
        .unstack(fill_value=0)
        .reindex(columns=COST_TYPE_IDS, fill_value=0)
    )

    cost_type_pct = cost_type_counts.div(cost_type_counts.sum(axis=1), axis=0).fillna(0)
    cost_type_pct.columns = [f'pct_cost_type_{cost_type_id}' for cost_type_id in cost_type_pct.columns]
    cost_type_pct = cost_type_pct.reset_index()

    weekly_features = weekly_summary.merge(
        cost_type_pct,
        on=['merchant_id', 'week_of_year'],
        how='left',
    )

    merchants = pd.Index(sorted(df_local['merchant_id'].dropna().unique()), name='merchant_id')
    weeks = pd.Index(range(1, 53), name='week_of_year')
    full_grid = pd.MultiIndex.from_product([merchants, weeks], names=['merchant_id', 'week_of_year']).to_frame(index=False)

    weekly_features = full_grid.merge(
        weekly_features,
        on=['merchant_id', 'week_of_year'],
        how='left',
    )

    pct_cols = [f'pct_cost_type_{cost_type_id}' for cost_type_id in COST_TYPE_IDS]
    weekly_features[pct_cols] = weekly_features[pct_cols].fillna(0.0)
    weekly_features['total_transaction_count'] = weekly_features['total_transaction_count'].fillna(0).astype(int)
    weekly_features['total_transaction_value'] = weekly_features['total_transaction_value'].fillna(0.0)
    weekly_features['split_year'] = split_year

    ordered_cols = [
        'split_year',
        'week_of_year',
        'merchant_id',
        'cost_percent',
        'total_transaction_count',
        'total_transaction_value',
        'cost_percent_stdev',
        *pct_cols,
    ]
    weekly_features = weekly_features[ordered_cols].sort_values(['merchant_id', 'week_of_year']).reset_index(drop=True)

    return weekly_features

# Build weekly features for each MCC across all available years
mcc_data = {
    5499: df_5499,
    4121: df_4121,
    5411: df_5411,
    5812: df_5812,
}

weekly_features_by_mcc = {}

for mcc, mcc_df in mcc_data.items():
    years = sorted(pd.to_datetime(mcc_df['date']).dt.year.dropna().unique().tolist())
    yearly_frames = [build_weekly_merchant_features(mcc_df, y) for y in years]
    weekly_mcc = pd.concat(yearly_frames, ignore_index=True) if yearly_frames else pd.DataFrame()
    weekly_features_by_mcc[mcc] = weekly_mcc
    print(f'MCC {mcc}: years={years}, weekly feature shape={weekly_mcc.shape}')

# Convenience variables
weekly_features_5499 = weekly_features_by_mcc[5499]
weekly_features_4121 = weekly_features_by_mcc[4121]
weekly_features_5411 = weekly_features_by_mcc[5411]
weekly_features_5812 = weekly_features_by_mcc[5812]

weekly_features_5499.head()

MCC 5499: years=[2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019], weekly feature shape=(3120, 68)
MCC 4121: years=[2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019], weekly feature shape=(401336, 68)
MCC 5411: years=[2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019], weekly feature shape=(1400412, 68)
MCC 5812: years=[2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019], weekly feature shape=(1185548, 68)


,split_year,week_of_year,merchant_id,cost_percent,total_transaction_count,total_transaction_value,cost_percent_stdev,pct_cost_type_1,pct_cost_type_2,pct_cost_type_3,...,pct_cost_type_52,pct_cost_type_53,pct_cost_type_54,pct_cost_type_55,pct_cost_type_56,pct_cost_type_57,pct_cost_type_58,pct_cost_type_59,pct_cost_type_60,pct_cost_type_61
0,2010,1,5248,1.196883,22,1277.500000,0.792854,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2010,2,5248,1.867319,16,736.200012,0.290180,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2010,3,5248,1.802594,14,842.450012,0.226407,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2010,4,5248,1.726511,23,1404.479980,0.483743,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2010,5,5248,1.792267,12,602.729980,0.230355,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 5. Comprehensive Testing Framework (Step-by-Step)

We will implement this in phases so each part is explainable and testable.

Phase 1 in this section:
1. Define onboarding episodes (1-month context -> 3-month horizon).
2. Build leakage-safe train/validation/test split assignments.
3. Produce episode tables by MCC for downstream model benchmarking.

In [28]:
# Step 1: shared protocol constants for episode-based benchmarking
CONTEXT_WEEKS = 4
HORIZON_WEEKS = 12
TOTAL_EPISODE_WEEKS = CONTEXT_WEEKS + HORIZON_WEEKS

MIN_CONTEXT_NON_NULL = 2
MIN_HORIZON_NON_NULL = 8

# Runtime controls
EPISODE_STRIDE_WEEKS = 1  # 1 = every possible onboarding week
MAX_QUERY_EPISODES_PER_MCC = 2000  # cap for Phase 2 evaluation runtime
RANDOM_SEED = 42

print('Episode protocol loaded')
print(f'Context weeks: {CONTEXT_WEEKS}, Horizon weeks: {HORIZON_WEEKS}')
print(f'Coverage rules: context >= {MIN_CONTEXT_NON_NULL}, horizon >= {MIN_HORIZON_NON_NULL}')
print(f'Episode stride: {EPISODE_STRIDE_WEEKS}, max query episodes per MCC: {MAX_QUERY_EPISODES_PER_MCC}')

Episode protocol loaded
Context weeks: 4, Horizon weeks: 12
Coverage rules: context >= 2, horizon >= 8
Episode stride: 1, max query episodes per MCC: 2000


### Step 2. Create Onboarding Episodes

Each episode simulates one merchant onboarding event:
- Context: 4 weeks from a chosen start week.
- Horizon: next 12 weeks.
- Merchant can appear at any valid start week (rolling windows).

This gives us many realistic test cases instead of only January starts.

In [29]:
# Step 2 code: build compact onboarding episode table from weekly merchant features
def build_episode_table(weekly_df, mcc):
    d = weekly_df.copy()
    d['split_year'] = pd.to_numeric(d['split_year'], errors='coerce').astype('Int64')
    d['week_of_year'] = pd.to_numeric(d['week_of_year'], errors='coerce').astype('Int64')
    d = d.sort_values(['merchant_id', 'split_year', 'week_of_year']).reset_index(drop=True)

    rows = []
    for merchant_id, g in d.groupby('merchant_id'):
        g = g.sort_values(['split_year', 'week_of_year']).reset_index(drop=True)
        n = len(g)

        if n < TOTAL_EPISODE_WEEKS:
            continue

        for i in range(0, n - TOTAL_EPISODE_WEEKS + 1, EPISODE_STRIDE_WEEKS):
            w = g.iloc[i:i + TOTAL_EPISODE_WEEKS]
            context = w.iloc[:CONTEXT_WEEKS]
            horizon = w.iloc[CONTEXT_WEEKS:]

            context_non_null = int(context['cost_percent'].notna().sum())
            horizon_non_null = int(horizon['cost_percent'].notna().sum())
            context_txn_total = int(context['total_transaction_count'].sum())
            horizon_txn_total = int(horizon['total_transaction_count'].sum())

            coverage_ok = (
                (context_non_null >= MIN_CONTEXT_NON_NULL)
                and (horizon_non_null >= MIN_HORIZON_NON_NULL)
            )
            if not coverage_ok:
                continue

            rows.append({
                'mcc': int(mcc),
                'merchant_id': int(merchant_id),
                'start_split_year': int(context['split_year'].iloc[0]),
                'start_week_of_year': int(context['week_of_year'].iloc[0]),
                'context_non_null': context_non_null,
                'horizon_non_null': horizon_non_null,
                'context_txn_total': context_txn_total,
                'horizon_txn_total': horizon_txn_total,
                'coverage_ok': True,
            })

    return pd.DataFrame(rows)

# Build episodes for all MCCs
episode_tables_by_mcc = {}
for mcc, wf in weekly_features_by_mcc.items():
    ep = build_episode_table(wf, mcc)
    episode_tables_by_mcc[mcc] = ep
    ok_count = int(ep['coverage_ok'].sum()) if not ep.empty else 0
    print(f'MCC {mcc}: episodes={len(ep):,}, coverage_ok={ok_count:,}')

episodes_all = pd.concat(episode_tables_by_mcc.values(), ignore_index=True) if episode_tables_by_mcc else pd.DataFrame()
print('\nTotal episodes across MCCs:', len(episodes_all))
episodes_all.head()

MCC 5499: episodes=3,006, coverage_ok=3,006
MCC 4121: episodes=39,406, coverage_ok=39,406
MCC 5411: episodes=259,600, coverage_ok=259,600
MCC 5812: episodes=141,253, coverage_ok=141,253

Total episodes across MCCs: 443265


,mcc,merchant_id,start_split_year,start_week_of_year,context_non_null,horizon_non_null,context_txn_total,horizon_txn_total,coverage_ok
0,5499,5248,2010,1,4,12,75,143,True
1,5499,5248,2010,2,4,12,65,159,True
2,5499,5248,2010,3,4,12,63,164,True
3,5499,5248,2010,4,4,12,61,166,True
4,5499,5248,2010,5,4,12,46,165,True


### Step 3. Assign Train/Validation/Test Episodes

We split by episode start year to preserve temporal realism:
- Train: early years
- Validation: middle years
- Test: latest years

This split is applied per MCC so each MCC keeps representation in every phase when possible.

In [30]:
# Step 3 code: assign episode splits and report cohort sizes
def assign_episode_split(ep_df):
    if ep_df.empty:
        return ep_df

    d = ep_df.copy()
    years = sorted(d['start_split_year'].dropna().unique().tolist())
    if len(years) < 3:
        # Fallback if limited years exist: last year as test, previous as validation
        test_year = years[-1]
        val_year = years[-2] if len(years) >= 2 else years[-1]
        train_years = [y for y in years if y not in [val_year, test_year]]
    else:
        train_years = years[:-2]
        val_year = years[-2]
        test_year = years[-1]

    d['episode_split'] = np.where(
        d['start_split_year'].isin(train_years), 'train',
        np.where(d['start_split_year'] == val_year, 'validation', 'test')
    )
    return d

episodes_all = assign_episode_split(episodes_all)

split_summary = (
    episodes_all.groupby(['mcc', 'episode_split'], as_index=False)
    .agg(
        episodes=('merchant_id', 'count'),
        merchants=('merchant_id', 'nunique'),
        coverage_ok_episodes=('coverage_ok', 'sum'),
    )
    .sort_values(['mcc', 'episode_split'])
    .reset_index(drop=True)
)

print('Episode split summary by MCC')
display(split_summary)

episodes_all.head()

Episode split summary by MCC


,mcc,episode_split,episodes,merchants,coverage_ok_episodes
0,4121,test,2656,97,2656
1,4121,train,32390,110,32390
2,4121,validation,4360,99,4360
3,5411,test,16680,745,16680
4,5411,train,215091,886,215091
5,5411,validation,27829,774,27829
6,5499,test,198,6,198
7,5499,train,2496,6,2496
8,5499,validation,312,6,312
9,5812,test,9136,459,9136


,mcc,merchant_id,start_split_year,start_week_of_year,context_non_null,horizon_non_null,context_txn_total,horizon_txn_total,coverage_ok,episode_split
0,5499,5248,2010,1,4,12,75,143,True,train
1,5499,5248,2010,2,4,12,65,159,True,train
2,5499,5248,2010,3,4,12,63,164,True,train
3,5499,5248,2010,4,4,12,61,166,True,train
4,5499,5248,2010,5,4,12,46,165,True,train


### Step 4. Validation Readiness Checks

Before fitting any model, confirm that each split has enough valid episodes after coverage filtering.

In the next section, we will implement:
1. Leakage-safe neighbor search per episode.
2. Composite merchant construction.
3. Unified SARIMA and supervised benchmarking with guarded calibration.

In [31]:
# Step 4 code: filter to valid episodes and prepare Phase 2 inputs
if episodes_all.empty:
    raise ValueError('No episodes built. Run Steps 1-3 first.')

episodes_valid = episodes_all[episodes_all['coverage_ok']].copy()

readiness = (
    episodes_valid.groupby(['mcc', 'episode_split'], as_index=False)
    .agg(
        valid_episodes=('merchant_id', 'count'),
        valid_merchants=('merchant_id', 'nunique'),
    )
    .sort_values(['mcc', 'episode_split'])
    .reset_index(drop=True)
)

print('Valid episode readiness by MCC and split')
display(readiness)

# Keep only columns needed for Phase 2
episode_working_cols = [
    'mcc', 'merchant_id', 'start_split_year', 'start_week_of_year',
    'episode_split', 'context_non_null', 'horizon_non_null',
    'context_txn_total', 'horizon_txn_total',
]
episodes_working = episodes_valid[episode_working_cols].reset_index(drop=True)

print('\nWorking episodes table shape:', episodes_working.shape)
episodes_working.head(10)

Valid episode readiness by MCC and split


,mcc,episode_split,valid_episodes,valid_merchants
0,4121,test,2656,97
1,4121,train,32390,110
2,4121,validation,4360,99
3,5411,test,16680,745
4,5411,train,215091,886
5,5411,validation,27829,774
6,5499,test,198,6
7,5499,train,2496,6
8,5499,validation,312,6
9,5812,test,9136,459



Working episodes table shape: (443265, 9)


,mcc,merchant_id,start_split_year,start_week_of_year,episode_split,context_non_null,horizon_non_null,context_txn_total,horizon_txn_total
0,5499,5248,2010,1,train,4,12,75,143
1,5499,5248,2010,2,train,4,12,65,159
2,5499,5248,2010,3,train,4,12,63,164
3,5499,5248,2010,4,train,4,12,61,166
4,5499,5248,2010,5,train,4,12,46,165
5,5499,5248,2010,6,train,4,12,43,165
6,5499,5248,2010,7,train,4,12,39,166
7,5499,5248,2010,8,train,4,12,39,168
8,5499,5248,2010,9,train,4,12,40,181
9,5499,5248,2010,10,train,4,12,50,185


## 6. Phase 2: Time-Aware k-NN and Composite Training

Phase 2 objective: for each onboarding episode, find similar merchants that are already in the system at that onboarding point, then train on composite pre-context history.

We do this in four steps:
1. Define helper functions to extract episode windows and vectors.
2. Build cached merchant panel metadata for fast time-aware neighbor filtering.
3. Run k-NN matching on the same context calendar window.
4. Train on composite history, predict context, apply guarded adjustment, and score on post-context horizon.

### Step 5. Helper Functions for Episode Windows and Vectors

Each episode is represented by:
- A 16-week window (4 context + 12 horizon)
- A context feature vector used by k-NN
- A horizon target trajectory used to build composite priors

In [21]:
# Step 5 code: helper functions and feature definitions
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances

K_NEIGHBORS_PHASE2 = 5

# Features used for k-NN similarity in the 4-week context window
SIMILARITY_FEATURE_COLS = [
    'cost_percent',
    'total_transaction_count',
    'total_transaction_value',
    'cost_percent_stdev',
] + [f'pct_cost_type_{i}' for i in COST_TYPE_IDS]

def build_week_index(df_weekly):
    """Create a fast lookup from (split_year, week_of_year) -> row index per merchant panel."""
    keys = list(zip(df_weekly['split_year'].tolist(), df_weekly['week_of_year'].tolist()))
    return {k: i for i, k in enumerate(keys)}

def extract_episode_window(df_weekly, start_year, start_week, total_weeks=TOTAL_EPISODE_WEEKS):
    """Extract a contiguous episode window by start (year, week) in the merchant panel."""
    idx_map = build_week_index(df_weekly)
    start_key = (int(start_year), int(start_week))
    if start_key not in idx_map:
        return None

    start_i = idx_map[start_key]
    end_i = start_i + total_weeks
    if end_i > len(df_weekly):
        return None

    w = df_weekly.iloc[start_i:end_i].copy()
    if len(w) != total_weeks:
        return None
    return w

def episode_context_vector(window_df, feature_cols=SIMILARITY_FEATURE_COLS):
    """Flatten first 4 context rows into a 1D vector for k-NN."""
    context = window_df.iloc[:CONTEXT_WEEKS].copy()
    context = context[feature_cols].fillna(0.0)
    return context.to_numpy(dtype=float).reshape(-1)

def episode_horizon_target(window_df, target_col='cost_percent'):
    """Return 12-week horizon target array."""
    horizon = window_df.iloc[CONTEXT_WEEKS:CONTEXT_WEEKS + HORIZON_WEEKS].copy()
    return horizon[target_col].astype(float).to_numpy()

print('Helper functions ready')
print('Similarity features:', len(SIMILARITY_FEATURE_COLS))

Helper functions ready
Similarity features: 65


### Step 6. Build Time-Aware Merchant Pool Helpers

For each onboarding episode, the neighbor pool must be merchants that are already in the system at that onboarding point.

Rules implemented here:
- Candidate neighbors are from the same MCC and not the onboarding merchant.
- Candidate neighbors must have the same 4-week calendar context window as the onboarding merchant.
- Candidate neighbors must have non-zero history before the onboarding context start week.
- Model training data is the weighted composite history from the start of history up to the week before context starts.

In [32]:
# Step 6 code: cached helper structures/functions for time-aware onboarding neighbor selection
def yw_to_ord(year, week):
    return int(year) * 100 + int(week)

def before_start_mask(df_panel, start_year, start_week):
    return (
        (df_panel['split_year'] < int(start_year))
        | ((df_panel['split_year'] == int(start_year)) & (df_panel['week_of_year'] < int(start_week)))
    )

def build_merchant_panels(weekly_features_by_mcc):
    panels = {}
    for mcc, wf in weekly_features_by_mcc.items():
        w = wf.copy().sort_values(['merchant_id', 'split_year', 'week_of_year']).reset_index(drop=True)
        mcc_panels = {}
        for mid, g in w.groupby('merchant_id'):
            panel = g.sort_values(['split_year', 'week_of_year']).reset_index(drop=True)
            key_tuples = list(zip(panel['split_year'].astype(int), panel['week_of_year'].astype(int)))
            key_map = {k: i for i, k in enumerate(key_tuples)}
            key_set = set(key_tuples)

            active = panel[panel['total_transaction_count'] > 0]
            if active.empty:
                first_active_ord = None
            else:
                first_active_ord = int((active['split_year'].astype(int) * 100 + active['week_of_year'].astype(int)).min())

            mcc_panels[int(mid)] = {
                'panel': panel,
                'key_map': key_map,
                'key_set': key_set,
                'first_active_ord': first_active_ord,
            }

        panels[int(mcc)] = mcc_panels
    return panels

def panel_has_context_keys(meta, context_keys):
    return set(context_keys).issubset(meta['key_set'])

def context_vector_from_keys(meta, context_keys, feature_cols=SIMILARITY_FEATURE_COLS):
    key_map = meta['key_map']
    if not all(k in key_map for k in context_keys):
        return None
    idx = [key_map[k] for k in context_keys]
    context = meta['panel'].iloc[idx][feature_cols].fillna(0.0)
    return context.to_numpy(dtype=float).reshape(-1)

def merchant_is_in_system(meta, start_year, start_week):
    first_active_ord = meta['first_active_ord']
    if first_active_ord is None:
        return False
    return first_active_ord < yw_to_ord(start_year, start_week)

def get_context_keys_from_window(window_df):
    c = window_df.iloc[:CONTEXT_WEEKS]
    return list(zip(c['split_year'].astype(int), c['week_of_year'].astype(int)))

def weighted_nanmean_2d(values_2d, weights_1d):
    out = []
    for j in range(values_2d.shape[1]):
        col = values_2d[:, j]
        ok = np.isfinite(col)
        if not ok.any():
            out.append(np.nan)
            continue
        out.append(np.average(col[ok], weights=weights_1d[ok]))
    return np.array(out, dtype=float)

def guarded_linear_adjust(raw_context_pred, actual_context, raw_full_pred, slope_clip=(0.5, 1.5), intercept_sigma=2.0):
    x = np.asarray(raw_context_pred, dtype=float)
    y = np.asarray(actual_context, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)

    if mask.sum() < 3 or np.nanstd(x[mask]) < 1e-10:
        return raw_full_pred.copy(), {'used': False, 'slope': np.nan, 'intercept': np.nan}

    X = np.column_stack([x[mask], np.ones(mask.sum())])
    slope, intercept = np.linalg.lstsq(X, y[mask], rcond=None)[0]

    slope = float(np.clip(slope, slope_clip[0], slope_clip[1]))
    y_std = float(np.nanstd(y[mask]))
    if y_std > 0:
        bound = intercept_sigma * y_std
        intercept = float(np.clip(intercept, -bound, bound))
    else:
        intercept = 0.0

    adjusted = slope * np.asarray(raw_full_pred, dtype=float) + intercept
    return adjusted, {'used': True, 'slope': slope, 'intercept': intercept}

merchant_panels_by_mcc = build_merchant_panels(weekly_features_by_mcc)
print('Merchant panels built for MCCs:', sorted(merchant_panels_by_mcc.keys()))
for mcc, mp in merchant_panels_by_mcc.items():
    print(f'MCC {mcc}: merchants in panel dictionary = {len(mp):,}')

Merchant panels built for MCCs: [4121, 5411, 5499, 5812]
MCC 5499: merchants in panel dictionary = 6
MCC 4121: merchants in panel dictionary = 3,053
MCC 5411: merchants in panel dictionary = 7,996
MCC 5812: merchants in panel dictionary = 6,484


### Step 7. Time-Aware k-NN, Composite-History Training, and Guarded Evaluation

For each query episode (validation/test):
1. Build neighbors from merchants already in-system at context start, using the same context weeks.
2. Build weighted composite history from start-of-history to pre-context week.
3. Train a simple forecasting model on that composite history.
4. Predict context weeks first, apply guarded linear adjustment, then score on the 12-week post-context horizon.

In [ ]:
# Step 7 code: time-aware neighbors + composite-history training + guarded horizon evaluation
def knn_time_aware_onboarding(query_row, merchant_panels_by_mcc, k=K_NEIGHBORS_PHASE2, eps=1e-9):
    mcc = int(query_row.mcc)
    merchant_id = int(query_row.merchant_id)
    start_year = int(query_row.start_split_year)
    start_week = int(query_row.start_week_of_year)

    mcc_panels = merchant_panels_by_mcc.get(mcc, {})
    query_meta = mcc_panels.get(merchant_id)
    if query_meta is None:
        return None

    query_panel = query_meta['panel']
    w_query = extract_episode_window(query_panel, start_year, start_week, total_weeks=TOTAL_EPISODE_WEEKS)
    if w_query is None:
        return None

    context_keys = get_context_keys_from_window(w_query)
    q_vec = context_vector_from_keys(query_meta, context_keys, SIMILARITY_FEATURE_COLS)
    if q_vec is None:
        return None

    q_context_actual = w_query.iloc[:CONTEXT_WEEKS]['cost_percent'].astype(float).to_numpy()
    q_horizon_actual = w_query.iloc[CONTEXT_WEEKS:CONTEXT_WEEKS + HORIZON_WEEKS]['cost_percent'].astype(float).to_numpy()

    # Candidate pool: same MCC, not query merchant, has same context weeks, already in-system before context start
    pool_rows = []
    for cand_mid, cand_meta in mcc_panels.items():
        if int(cand_mid) == merchant_id:
            continue
        if not panel_has_context_keys(cand_meta, context_keys):
            continue
        if not merchant_is_in_system(cand_meta, start_year, start_week):
            continue

        cand_vec = context_vector_from_keys(cand_meta, context_keys, SIMILARITY_FEATURE_COLS)
        if cand_vec is None:
            continue

        pool_rows.append({
            'merchant_id': int(cand_mid),
            'context_vector': cand_vec,
            'meta': cand_meta,
        })

    if len(pool_rows) == 0:
        return None

    pool = pd.DataFrame(pool_rows)
    X_pool = np.vstack(pool['context_vector'].values)
    x_query = q_vec.reshape(1, -1)

    scaler = StandardScaler()
    X_pool_z = scaler.fit_transform(X_pool)
    x_query_z = scaler.transform(x_query)

    dists = euclidean_distances(x_query_z, X_pool_z).reshape(-1)
    pool['distance'] = dists

    k_local = min(k, len(pool))
    nbrs = pool.nsmallest(k_local, 'distance').copy().reset_index(drop=True)
    nbrs['raw_weight'] = 1.0 / (nbrs['distance'] + eps)
    nbrs['weight'] = nbrs['raw_weight'] / nbrs['raw_weight'].sum()

    # Build composite history from start-of-history to week before context start
    mcc_keys = query_panel[['split_year', 'week_of_year']].drop_duplicates().sort_values(['split_year', 'week_of_year'])
    mcc_keys['key_ord'] = mcc_keys['split_year'].astype(int) * 100 + mcc_keys['week_of_year'].astype(int)
    start_ord = yw_to_ord(start_year, start_week)
    hist_keys_df = mcc_keys[mcc_keys['key_ord'] < start_ord][['split_year', 'week_of_year']]
    hist_keys = list(zip(hist_keys_df['split_year'].astype(int), hist_keys_df['week_of_year'].astype(int)))

    if len(hist_keys) < 8:
        return None

    neighbor_hist_vals = []
    for row_n in nbrs.itertuples(index=False):
        meta = row_n.meta
        p = meta['panel'][['split_year', 'week_of_year', 'cost_percent']].copy()
        p['k'] = list(zip(p['split_year'].astype(int), p['week_of_year'].astype(int)))
        mp = dict(zip(p['k'], p['cost_percent']))
        vals = np.array([mp.get(kh, np.nan) for kh in hist_keys], dtype=float)
        neighbor_hist_vals.append(vals)

    hist_mat = np.vstack(neighbor_hist_vals)
    weights = nbrs['weight'].to_numpy(dtype=float)
    composite_history = weighted_nanmean_2d(hist_mat, weights)

    valid_hist = np.isfinite(composite_history)
    if valid_hist.sum() < 8:
        return None

    # Baseline forecast on composite history
    y_hist = composite_history[valid_hist]
    t_hist = np.arange(len(y_hist), dtype=float)

    if len(y_hist) >= 8 and np.nanstd(y_hist) > 1e-10:
        slope, intercept = np.polyfit(t_hist, y_hist, 1)
    else:
        slope, intercept = 0.0, float(np.nanmean(y_hist))

    total_pred_len = CONTEXT_WEEKS + HORIZON_WEEKS
    t_future = np.arange(len(y_hist), len(y_hist) + total_pred_len, dtype=float)
    raw_full_pred = slope * t_future + intercept

    raw_context_pred = raw_full_pred[:CONTEXT_WEEKS]
    raw_horizon_pred = raw_full_pred[CONTEXT_WEEKS:]

    adj_full_pred, adj_info = guarded_linear_adjust(
        raw_context_pred=raw_context_pred,
        actual_context=q_context_actual,
        raw_full_pred=raw_full_pred,
    )
    adj_context_pred = adj_full_pred[:CONTEXT_WEEKS]
    adj_horizon_pred = adj_full_pred[CONTEXT_WEEKS:]

    return {
        'mcc': mcc,
        'merchant_id': merchant_id,
        'start_split_year': start_year,
        'start_week_of_year': start_week,
        'episode_split': query_row.episode_split,
        'k_used': int(k_local),
        'neighbors': nbrs[['merchant_id', 'distance', 'weight']].copy(),
        'context_keys': context_keys,
        'composite_history': composite_history,
        'context_actual': q_context_actual,
        'horizon_actual': q_horizon_actual,
        'context_pred_raw': raw_context_pred,
        'context_pred_adj': adj_context_pred,
        'horizon_pred_raw': raw_horizon_pred,
        'horizon_pred_adj': adj_horizon_pred,
        'adj_info': adj_info,
    }

query_episodes = episodes_working[episodes_working['episode_split'].isin(['validation', 'test'])].copy()

# Runtime cap: sample up to N onboarding episodes per MCC
query_episodes = query_episodes.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
query_episodes['rank_within_mcc'] = query_episodes.groupby('mcc').cumcount() + 1
query_episodes = query_episodes[query_episodes['rank_within_mcc'] <= MAX_QUERY_EPISODES_PER_MCC].drop(columns=['rank_within_mcc']).reset_index(drop=True)

print('Query episodes after cap:', len(query_episodes))
print(query_episodes.groupby('mcc').size().rename('n').to_string())

composite_rows = []
composite_objects = []

for row in query_episodes.itertuples(index=False):
    out = knn_time_aware_onboarding(row, merchant_panels_by_mcc, k=K_NEIGHBORS_PHASE2)
    if out is None:
        continue

    context_mae_raw = np.nanmean(np.abs(out['context_actual'] - out['context_pred_raw']))
    context_mae_adj = np.nanmean(np.abs(out['context_actual'] - out['context_pred_adj']))
    horizon_mae_raw = np.nanmean(np.abs(out['horizon_actual'] - out['horizon_pred_raw']))
    horizon_mae_adj = np.nanmean(np.abs(out['horizon_actual'] - out['horizon_pred_adj']))

    composite_rows.append({
        'mcc': out['mcc'],
        'merchant_id': out['merchant_id'],
        'start_split_year': out['start_split_year'],
        'start_week_of_year': out['start_week_of_year'],
        'episode_split': out['episode_split'],
        'k_used': out['k_used'],
        'adj_used': bool(out['adj_info']['used']),
        'context_mae_raw': float(context_mae_raw),
        'context_mae_adj': float(context_mae_adj),
        'horizon_mae_raw_12w': float(horizon_mae_raw),
        'horizon_mae_adj_12w': float(horizon_mae_adj),
    })
    composite_objects.append(out)

composite_prior_eval = pd.DataFrame(composite_rows)
print('Time-aware composites built:', len(composite_prior_eval))

if not composite_prior_eval.empty:
    summary = (
        composite_prior_eval.groupby(['mcc', 'episode_split'], as_index=False)
        .agg(
            episodes=('merchant_id', 'count'),
            mean_k=('k_used', 'mean'),
            context_mae_raw=('context_mae_raw', 'mean'),
            context_mae_adj=('context_mae_adj', 'mean'),
            horizon_mae_raw_12w=('horizon_mae_raw_12w', 'mean'),
            horizon_mae_adj_12w=('horizon_mae_adj_12w', 'mean'),
        )
        .sort_values(['mcc', 'episode_split'])
        .reset_index(drop=True)
    )
    display(summary)

composite_prior_eval.head(10)

Query episodes after cap: 6510
mcc
4121    2000
5411    2000
5499     510
5812    2000


### Step 8. Inspect One Example: Neighbors, Context Fit, and Horizon Forecast

This check shows:
- Which merchants were selected as in-system neighbors
- How raw and adjusted predictions fit the context window
- How both versions perform on the next 12 weeks

In [ ]:
# Step 8 code: inspect a sample query episode with context fit and horizon forecast
if len(composite_objects) == 0:
    print('No composite objects available. Run Step 7 first.')
else:
    ex = composite_objects[0]
    print('Sample query episode')
    print({
        'mcc': ex['mcc'],
        'merchant_id': ex['merchant_id'],
        'start_split_year': ex['start_split_year'],
        'start_week_of_year': ex['start_week_of_year'],
        'k_used': ex['k_used'],
        'guarded_adjustment_used': ex['adj_info']['used'],
        'guard_slope': ex['adj_info']['slope'],
        'guard_intercept': ex['adj_info']['intercept'],
    })

    print('\nNeighbors (time-aware pool):')
    display(ex['neighbors'])

    context_df = pd.DataFrame({
        'context_pos': np.arange(1, CONTEXT_WEEKS + 1),
        'actual': ex['context_actual'],
        'pred_raw': ex['context_pred_raw'],
        'pred_adj': ex['context_pred_adj'],
    })
    horizon_df = pd.DataFrame({
        'horizon_pos': np.arange(1, HORIZON_WEEKS + 1),
        'actual': ex['horizon_actual'],
        'pred_raw': ex['horizon_pred_raw'],
        'pred_adj': ex['horizon_pred_adj'],
    })

    display(context_df)
    display(horizon_df.head(12))

    plt.figure(figsize=(12, 4))
    plt.plot(context_df['context_pos'], context_df['actual'], marker='o', label='Context actual')
    plt.plot(context_df['context_pos'], context_df['pred_raw'], marker='s', label='Context pred raw')
    plt.plot(context_df['context_pos'], context_df['pred_adj'], marker='^', label='Context pred adjusted')
    plt.title('Context Window Fit (Used for Guarded Adjustment)')
    plt.xlabel('Context week position (1-4)')
    plt.ylabel('cost_percent')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 4))
    plt.plot(horizon_df['horizon_pos'], horizon_df['actual'], marker='o', label='Horizon actual')
    plt.plot(horizon_df['horizon_pos'], horizon_df['pred_raw'], marker='s', label='Horizon pred raw')
    plt.plot(horizon_df['horizon_pos'], horizon_df['pred_adj'], marker='^', label='Horizon pred adjusted')
    plt.title('Post-Context 12-Week Horizon Forecast')
    plt.xlabel('Horizon week position (1-12)')
    plt.ylabel('cost_percent')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()